In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Hotel Dataset — 30 Data Cleaning Questions

In [2]:
df = pd.read_csv("C:\\Users\\MY PC\\Downloads\\Hotel_booking_dataset - Hotel_booking_dataset.csv")
df.head(5)

,Booking ID,Customer Name,Booking Date,City,Room Type,Booking Channel,Customer Segment,Gender,Age,Room Nights,...,Rating,Guests,Is Member,Cancelled,Payment Method,Gross Amount,Discount Amount,Final Amount,Hotel Cost,Profit
0,BKG01888,Customer_1888,2025-05-17 0:00:00,Mumbai,Suite,Online,Premium,Male,39,4,...,1.4,2,1,0,Credit Card,31674.48,2096.850576,29577.629420,16901.386250,12676.243170
1,BKG00789,Customer_0789,2025-07-29 0:00:00,Bengaluru,Suite,Walk-in,Corporate,Male,25,1,...,2.5,2,1,0,Credit Card,4822.39,735.414475,4086.975525,2746.904163,1340.071362
2,BKG00757,Customer_0757,2025-10-19 0:00:00,Pune,Suite,Corporate,Corporate,Other,50,4,...,1.2,1,0,1,Credit Card,66334.52,15575.345300,50759.174700,40212.657470,10546.517240
3,BKG01079,Customer_1079,2025-12-14 0:00:00,Mumbai,Deluxe,Walk-in,Regular,Male,53,3,...,2.6,6,0,0,Debit Card,19255.92,5328.113064,13927.806940,9639.061764,4288.745172
4,BKG00218,Customer_0218,2025-01-20 0:00:00,Delhi,Suite,Travel Agent,Budget,Male,38,NaN,...,4.2,4,1,0,Cash,15685.38,5160.490020,10524.889980,6545.581304,3979.308676


# 1. Remove leading/trailing spaces from column names

In [4]:
df.columns = df.columns.str.strip()
print(df.columns)

Index(['Booking ID', 'Customer Name', 'Booking Date', 'City', 'Room Type',
       'Booking Channel', 'Customer Segment', 'Gender', 'Age', 'Room Nights',
       'Room Rate', 'Discount %', 'Rating', 'Guests', 'Is Member', 'Cancelled',
       'Payment Method', 'Gross Amount', 'Discount Amount', 'Final Amount',
       'Hotel Cost', 'Profit'],
      dtype='object')


# 2. Standardize all date formats into one consistent format (YYYY-MM-DD)

In [5]:
df['Booking Date'] = pd.to_datetime(df['Booking Date'],errors ="coerce",dayfirst = True)
df['Booking Date'] = df['Booking Date'].dt.strftime('%Y-%m-%d')
df['Booking Date']

C:\Users\MY PC\AppData\Local\Temp\ipykernel_10104\239541544.py:1: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Booking Date'] = pd.to_datetime(df['Booking Date'],errors ="coerce",dayfirst = True)


0       2025-05-17
1       2025-07-29
2       2025-10-19
3       2025-12-14
4       2025-01-20
           ...    
2020    2025-11-27
2021    2025-10-22
2022    2025-07-11
2023    2025-01-24
2024    2025-04-06
Name: Booking Date, Length: 2025, dtype: object

# 3. Handle completely missing Customer Name values

In [6]:
df['Customer Name'] = df['Customer Name'].replace('N/A',pd.NA)
df['Customer Name'] = df['Customer Name'].fillna("unknown")
df['Customer Name'].dropna() 

0       Customer_1888
1       Customer_0789
2       Customer_0757
3       Customer_1079
4       Customer_0218
            ...      
2020    Customer_1738
2021    Customer_1193
2022    Customer_1210
2023    Customer_1060
2024    Customer_0642
Name: Customer Name, Length: 2025, dtype: object

# 4. Fix inconsistent Gender values
# Explanation: Gender appears as Male, FEMALE, male, F, M, Other, etc. Convert everything to
# a standard set such as Male, Female, Other.

In [7]:
df['Gender'].unique()

array(['Male', 'Other', 'Female', 'FEMALE', 'male', 'M', 'female', 'F'],
      dtype=object)

In [8]:
df['Gender'] = df['Gender'].replace({
                                'male':'Male',
                                'M'   :'Male',
                                'Female':'FEMALE',
                                'F':'FEMALE',
                                'female':'FEMALE'
})
df['Gender'].unique()

array(['Male', 'Other', 'FEMALE'], dtype=object)

## 5. Clean and standardize City names
## Explanation: Cities have typos and casing issues (mumbai, Mumbai , Bangalore, Hydrabad,
## blank). Correct spelling, remove extra spaces, and make casing consistent (e.g., Title Case).

In [9]:
df['City'].unique()

array(['Mumbai', 'Bengaluru', 'Pune', 'Delhi', 'Chennai', 'Hyderabad',
       nan, 'mumbai', 'delhi', 'Bangalore', 'MUMBAI', 'Hydrabad',
       'bangalore'], dtype=object)

In [10]:
df['City'] = df['City'].str.strip().str.title()
df['City'] = df['City'].replace({
    'mumbai':'MUMBAI',
    'Mumbai':'MUMBAI',
    'bangalore':'Bengaluru',
    'Bangalore':'Bengaluru',
    'Hydrabad':'Hyderabad',
    'delhi':'Delhi'
})
df = df[df['City'].isin(['MUMBAI', 'Bengaluru', 'Hyderabad', 'Pune','Delhi','Chennai',''])]
df['City'].unique()

array(['MUMBAI', 'Bengaluru', 'Pune', 'Delhi', 'Chennai', 'Hyderabad'],
      dtype=object)

In [27]:
df['City'].value_counts()

City
MUMBAI       275
Delhi        273
Pune         255
Hyderabad    241
Bengaluru    240
Chennai      229
Name: count, dtype: int64

# 6. Standardize Room Type categories
# Explanation: Room Type has Suite, Deluxe, delux, standard, exec, Unknown, and blanks.
# Map similar values to a clean list: Standard, Deluxe, Suite, Executive.

In [11]:
df['Room Type'].unique()

array(['Suite', 'Deluxe', 'Standard', 'Executive', nan, 'standard',
       'STANDARD', 'delux', 'exec', 'Unknown', 'suite'], dtype=object)

In [12]:
df['Room Type'] = df['Room Type'].replace({
    'suite':'Suite',
    'delux':'Deluxe',
    'Standard':'STANDARD',
    'standard':'STANDARD',
    'exec':'Executive'
    
})

df['Room Type'] = df['Room Type'].str.strip()
df = df[df['Room Type'].isin(['Standard', 'Deluxe', 'Suite', 'Executive'])]


In [13]:
df['Room Type'].unique()

array(['Suite', 'Deluxe', 'Executive'], dtype=object)

In [26]:
df['Room Type'].value_counts()

Room Type
Deluxe       544
Executive    499
Suite        470
Name: count, dtype: int64

## 7. Clean Booking Channel values
## Explanation: Values include Online, online, Walk-in, Travel Agent, TravelAgent, corp,
## Corporate. Unify them into a small consistent set (Online, Walk-in, Corporate, Travel Agent).

In [14]:
df['Booking Channel'].unique()

array(['Online', 'Walk-in', 'Corporate', 'Travel Agent', 'online',
       'walkin', 'ONLINE', 'TravelAgent', 'corp', 'travel agent'],
      dtype=object)

In [23]:
df['Booking Channel'] = df['Booking Channel'].replace({
    'ONLINE':'Online',
    'online':'Online',
    'walkin':'Walk-in',
    'corp':'Corporate',
    'travel agent':'Travel Agent'
    
})
df['Booking Channel'].unique()

array(['Online', 'Walk-in', 'Corporate', 'Travel Agent', 'TravelAgent'],
      dtype=object)

In [25]:
df['Booking Channel'].value_counts()

Booking Channel
Walk-in         410
Online          404
Travel Agent    360
Corporate       334
TravelAgent       5
Name: count, dtype: int64

## 8. Standardize Customer Segment
## Explanation: Segments appear as Premium, regular, REG, Budget , VIP, etc. Create a clean set
## such as Budget, Regular, Premium, Corporate, VIP and map everything to it.

In [16]:
df['Customer Segment'].unique()

array(['Premium', 'Corporate', 'Regular', 'Budget', 'regular', 'VIP',
       'REG', 'premium', 'corporate'], dtype=object)

In [18]:
df['Customer Segment'] = df['Customer Segment'].replace({
    'REG':'Regular',
    'regular':'Regular',
    'premium':'Premium',
    'corporate':'Corporate'
    
})
df['Customer Segment'].unique()

array(['Premium', 'Corporate', 'Regular', 'Budget', 'VIP'], dtype=object)

In [19]:
df['Customer Segment'].value_counts()

Customer Segment
Budget       402
Premium      387
Regular      366
Corporate    354
VIP            4
Name: count, dtype: int64

## 9. Fix Age column (invalid and non-numeric values)
## Explanation: Age has numbers, blanks, 30 yrs, 25 years, 45Y, and impossible values like 150
## or 100. Extract the number, convert to integer, and set unrealistic ages (e.g., <18 or >100) to
## missing.

In [20]:
df['Age'].unique()

array(['39', '25', '50', '53', '38', '65', '48', '47', '52', '44', '29',
       '49', '34', '33', '28', '62', '64', '21', '26', '57', '22', '59',
       '58', '54', '31', '24', '68', '32', '27', '36', '46', '100', '43',
       '20', nan, '19', '51', '30', '40', '18', '23', '61', '63', '45',
       '66', '69', '41', '67', '150', '55', '120', '42', '35', '56',
       'unknown', '37', '45Y', '60', '25 years', '30 yrs'], dtype=object)

In [52]:
df['Age'] = df['Age'].replace({
    "45Y":45,
    '25 years':25,
    '30 yrs':30,
    'unknown':'',
    '150':np.nan,
    '120':np.nan

})

In [51]:
df['Age'] =pd.to_numeric(df['Age'], errors ='coerce')
df['Age'].unique()

array([ 39.,  25.,  50.,  53.,  38.,  65.,  48.,  47.,  52.,  44.,  29.,
        49.,  34.,  33.,  28.,  62.,  64.,  21.,  26.,  57.,  22.,  59.,
        58.,  54.,  31.,  24.,  68.,  32.,  27.,  36.,  46., 100.,  43.,
        20.,  nan,  19.,  51.,  30.,  40.,  18.,  23.,  61.,  63.,  45.,
        66.,  69.,  41.,  67., 150.,  55., 120.,  42.,  35.,  56.,  37.,
        60.])

## 10. Clean Room Nights column
## Explanation: Values include numbers, N/A, 3 nights, 100, 30. Extract only the numeric part,
## convert to integer, and decide what to do with extreme values (e.g., 100 nights).

In [35]:
df['Room Nights'].unique()

array(['4', '1', '3', nan, '2', '5', '100', '7', '6', '3 nights', 'two',
       '30', '5 days', '50'], dtype=object)

In [40]:
df['Room Nights']= df['Room Nights'].replace({
    '3 nights':3,
    'two':2,
    '5 days':5,
    '100':np.nan,
    '50':np.nan
})
df['Room Nights']= pd.to_numeric(df['Room Nights'],errors = 'coerce')
df['Room Nights'].unique()

array([ 4.,  1.,  3., nan,  2.,  5.,  7.,  6., 30.])

In [42]:
df['Room Nights'].value_counts()

Room Nights
2.0     236
6.0     223
7.0     214
1.0     211
4.0     210
5.0     202
3.0     201
30.0      3
Name: count, dtype: int64

## 11. Fix Room Rate (mixed formats and extreme values)
## Explanation: Room Rate contains normal numbers, 15k, $8000, 12000 INR, 100000, 250000,
## and blanks. Convert text formats to numbers, remove currency symbols, and flag or handle
## unrealistic rates.

In [3]:
df['Room Rate']=df['Room Rate'].astype(str).str.strip()
df['Room Rate']

0        7918.62
1        4822.39
2       16583.63
3        6418.64
4        5228.46
          ...   
2020     4259.26
2021         15k
2022    13257.76
2023     9020.55
2024     1528.93
Name: Room Rate, Length: 2025, dtype: object

In [ ]:
df['Room Rate'] = df['Room Rate'].str.replace('$', '', regex=False)
df['Room Rate'] = df['Room Rate'].str.replace('INR', '', regex=False)
df['Room Rate'] = df['Room Rate'].str.replace('k', '000', regex=False)